# Macrocosm photo-z CNN — v4 (cleaned 600k, 16px)
Thin runner: downloads the **v4** data and calls `train()` in `photoz_cnn.py` (edit it to change the model).
Data = `catalog_v4` (quality-cleaned 600k, contiguous idx) + **sample_v4.7** (16×16×5 registered cutouts).


## 1. Setup + data
Clone the repo (code + v4 `splits/`), install deps, log into Google, download the catalog.
Runtime → Change runtime type → **GPU**.


In [ ]:
import os
os.environ['CUTOUT_SIZE'] = '16'        # sample_v4.7 is 16x16
![ -d /content/CNN-Model ] || git clone -b task-v4.5 https://github.com/Le-Wagon-Macrocosm/CNN-Model.git /content/CNN-Model
%cd /content/CNN-Model
!pip -q install mlflow
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q
!mkdir -p /content/data
# catalog_v4 (cleaned 600k, idx-aligned to sample_v4.7) -> saved as catalog_v1.parquet (the loader's expected name)
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.7/catalog_v4.parquet /content/data/catalog_v1.parquet
DATA_DIR = '/content/data'

## 1b. Image shards
`sample_v4.7` = 16px registered cutouts (cleaned 600k), ~1.5 GB — fits a **standard** runtime.


In [ ]:
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.7/images_*.npy /content/data/

## 2. Train
Uses the v4 `splits/` (train 554,628 / val 45,372). Paste the **MLflow token** to log; `None` = train only.
Edit `photoz_cnn.py` (the model cell) to try your own architecture.


In [ ]:
from photoz_cnn import train

metrics, model = train(
    data_dir='/content/data',
    crop=16,                 # sample_v4.7 is 16x16
    preproc='p99',           # per-band / p99 (best so far)
    N=None,                  # cap #train images if RAM-limited; None = all (~555k)
    batch=256, lr=3e-4, epochs=50,
    run_name='v4-run',
    experiment='oa',
    mlflow_token='PASTE_MLFLOW_API_TOKEN_HERE',
)
print(metrics)

## 3. (optional) 3-fold OOF outliers → GCS


In [ ]:
# from cv_outliers import run
# df = run(seed=0, data_dir='/content/data', crop=16, preproc='p99')